# Eval: Tropical Cyclone (TC) From Predictions (Detailed)

This notebook unrolls the prediction-driven TC workflow in Python steps.

Important prerequisite:
- `metview` executable must be installed and on `PATH`.
  The module imports `DataRetriever`, which depends on Metview for GRIB reference loading.


In [ ]:
from pathlib import Path
import shutil
import json
import numpy as np
import matplotlib.pyplot as plt

HAVE_METVIEW = shutil.which('metview') is not None
print('metview in PATH:', HAVE_METVIEW)

if HAVE_METVIEW:
    import eval.tc.plot_pdf_tc_from_predictions as tc
else:
    tc = None
    print('Install/load metview first, then re-run this cell.')


In [ ]:
# Step 1: configure paths
predictions_dir = Path('/path/to/predictions_folder')   # contains predictions_YYYYMMDD_stepXXX.nc
outdir = Path('/tmp/tc_eval')
run_label = 'my_run'
base_tc_dir = '/home/ecm5702/hpcperm/data/tc'

outdir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Step 2: discover and parse prediction files
if tc is not None:
    files = tc._discover_prediction_files(predictions_dir)
    print('found files:', len(files))
    for item in files[:5]:
        print(item)


In [ ]:
# Step 3: extract prediction vectors inside the TC domain
if tc is not None and predictions_dir.exists():
    files = tc._discover_prediction_files(predictions_dir)
    if files:
        pred_msl, pred_wind, days, steps, year, month = tc._extract_pred_vectors(files, tc.IDALIA_DOMAIN)
        print('pred_msl size:', pred_msl.size)
        print('pred_wind size:', pred_wind.size)
        print('days:', days)
        print('steps:', steps)
    else:
        print('No predictions_*.nc files matched expected naming.')


In [ ]:
# Step 4: inline histogram plots (prediction-only quick check)
if tc is not None and 'pred_msl' in locals():
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].hist(pred_msl, bins=tc.MSLP_BINS, density=True, alpha=0.8, color='royalblue')
    axs[0].set_title('Prediction MSLP PDF')
    axs[0].set_xlabel('hPa')

    axs[1].hist(pred_wind, bins=tc.WIND_BINS, density=True, alpha=0.8, color='darkorange')
    axs[1].set_title('Prediction 10m wind PDF')
    axs[1].set_xlabel('m/s')

    for ax in axs:
        ax.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Step 5: extreme-tail stats table (prediction curve only)
if tc is not None and 'pred_msl' in locals():
    tail = tc._extreme_tail_table(
        {run_label: (pred_msl, pred_wind)},
        mslp_range=tc.IDALIA_EXTREME_MSLP_RANGE,
        wind_gt=tc.IDALIA_EXTREME_WIND_MIN,
    )
    print(json.dumps(tail, indent=2))


In [ ]:
# Step 6: full reference comparison + PDF output (requires Metview + TC GRIB data)
RUN_FULL_TC = False

if tc is not None and RUN_FULL_TC:
    out_pdf = tc.run_tc_pdf_from_predictions(
        predictions_dir=str(predictions_dir),
        outdir=str(outdir),
        run_label=run_label,
        base_tc_dir=base_tc_dir,
        extra_reference_expids=[],
    )
    print('Saved PDF:', out_pdf)
    stats_path = Path(out_pdf).with_suffix('.stats.json')
    print('Stats JSON:', stats_path)
